In [ ]:
import os
import time
import numpy as np
import rasterio
from tqdm import tqdm
import gc

# File paths
area_tif_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Depth_Classification\LCMAP_CU_2021_V13_LCPRI.tif"
edge_tif_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\LCMAP_edges\LCMAP_2021_edges.tif"

# Output path (change to a user-writable directory)
output_dir = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Edge_Area_2021_1km"
os.makedirs(output_dir, exist_ok=True)

forest_output_path = os.path.join(output_dir, "Forest_Area_2021_1km.tif")
edge_output_path = os.path.join(output_dir, "Edge_Length_2021_1km.tif")

# Parameters
pixel_area = 30 * 30  # m^2
edge_length_per_edge = 30  # m
aggregation_factor = 33  # ~1km

# Load forest area map
with rasterio.open(area_tif_path) as src_area:
    area_data = src_area.read(1)
    area_transform = src_area.transform
    area_crs = src_area.crs

# Forest area map
forest_mask = area_data > 0
forest_area_map = forest_mask.astype(np.int16) * pixel_area
del forest_mask, area_data
gc.collect()

# Aggregation with progress
def block_sum_with_progress(data, block_size):
    h, w = data.shape
    new_h = h // block_size
    new_w = w // block_size
    result = np.zeros((new_h, new_w), dtype=np.int32)
    for i in tqdm(range(new_h), desc="Processing Rows"):
        for j in range(new_w):
            block = data[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
            result[i, j] = np.sum(block)
    return result

agg_forest_area = block_sum_with_progress(forest_area_map, aggregation_factor)
del forest_area_map
gc.collect()

agg_forest_area = np.nan_to_num(agg_forest_area, nan=0)

# Metadata
meta = {
    "driver": "GTiff",
    "height": agg_forest_area.shape[0],
    "width": agg_forest_area.shape[1],
    "count": 1,
    "dtype": "int32",
    "crs": area_crs,
    "transform": new_transform,
    "compress": "lzw"
}

# Save outputs
with rasterio.open(forest_output_path, "w", **meta) as dst:
    dst.write(agg_forest_area, 1)

del agg_forest_area
gc.collect()

# Load edge data
with rasterio.open(edge_tif_path) as src_edge:
    edge_data = src_edge.read(1)

# Count edges in binary
edge_count_map = np.vectorize(lambda x: bin(x).count("1"))(edge_data) * edge_length_per_edge

del edge_data
gc.collect()

agg_edge_length = block_sum_with_progress(edge_count_map, aggregation_factor)
del edge_count_map
gc.collect()

agg_edge_length = np.nan_to_num(agg_edge_length, nan=0)

# Define transform for 1km resolution
new_transform = rasterio.transform.Affine(
    area_transform.a * aggregation_factor, area_transform.b, area_transform.c,
    area_transform.d, area_transform.e * aggregation_factor, area_transform.f
)

# Metadata
meta = {
    "driver": "GTiff",
    "height": agg_forest_area.shape[0],
    "width": agg_forest_area.shape[1],
    "count": 1,
    "dtype": "int32",
    "crs": area_crs,
    "transform": new_transform,
    "compress": "lzw"
}

with rasterio.open(edge_output_path, "w", **meta) as dst:
    dst.write(agg_edge_length, 1)

del agg_edge_length
gc.collect()

print("✅ Forest area and edge length maps saved successfully.")
forest_output_path, edge_output_path